In [ ]:
import os, gc, time, json, warnings
import numpy as np
import pandas as pd
from itertools import combinations, product
warnings.filterwarnings("ignore")

from scipy.optimize import minimize
from scipy.special  import softmax

from sklearn.model_selection    import StratifiedKFold
from sklearn.preprocessing      import StandardScaler, label_binarize, normalize
from sklearn.decomposition      import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.feature_selection  import RFE
from sklearn.calibration        import CalibratedClassifierCV
from sklearn.metrics            import (accuracy_score, balanced_accuracy_score,
                                        f1_score, matthews_corrcoef,
                                        precision_score, recall_score,
                                        confusion_matrix, roc_auc_score,
                                        average_precision_score)
from sklearn.linear_model       import LogisticRegression
from sklearn.svm                import SVC, LinearSVC
from sklearn.neighbors          import KNeighborsClassifier
from sklearn.neural_network     import MLPClassifier
from sklearn.naive_bayes        import GaussianNB
from sklearn.ensemble           import (RandomForestClassifier, ExtraTreesClassifier,
                                        AdaBoostClassifier,
                                        HistGradientBoostingClassifier)
from sklearn.tree               import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_sample_weight, compute_class_weight
from lightgbm                   import LGBMClassifier
from xgboost                    import XGBClassifier
from catboost                   import CatBoostClassifier
from sklearn.feature_selection import (SelectKBest, f_classif,
                                       mutual_info_classif, SelectFromModel)

In [ ]:
# ==============================================================================
# CONFIG
# ==============================================================================
FEATURES_PATH = "/kaggle/input/datasets/hsharmaa/foodev-13299/features_prott5_2660.csv"   # TRAIN, full 1024-d
OUTPUT_DIR    = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

ID_COL       = "seq_id"
TARGET_COL   = "primary_label"
STRATIFY_COL = "secondary_label"
META_COLS    = [ID_COL, "primary_label", "secondary_label"]
CLASS_NAMES  = {0: "Non_EV", 1: "Milk_EV", 2: "Plant_EV"}

N_SPLITS       = 5
INNER_VAL_FRAC = 0.20
SEED           = 42

# --------------------------------------------------------------------------
# K SELECTION
# --------------------------------------------------------------------------

K_GRID   = [32, 64, 128, 256, 384, 512, 768]
WHITEN_K = 256     
PLS_K    = 120     



RECOMMENDED = [#"Full_1024",           # baseline
               "ANOVA_f",             # univariate filter (represents ANOVA/MI/corr)
               "LGBM_gain",           # embedded importance (best-performing selector)
               "RFE_logreg",          # wrapper
               "Whiten",              # geometry: near-Mahalanobis
               "Center_L2",           # geometry: cosine
               "PLSDA"]               # geometry: supervised projection

FULL = [#"Full_1024",
        "ANOVA_f", "MutualInfo", "Corr_target",          # filters
        "L1_logreg", "RF_importance", "LGBM_gain",        # embedded
        "RFE_logreg",                                     # wrapper
        "PCA", "Whiten", "Center_L2", "PLSDA"]            # transformation

RUN_METHODS = RECOMMENDED          

FAMILY = {#Full_1024": "baseline",
          "ANOVA_f": "selection", "MutualInfo": "selection", "Corr_target": "selection",
          "L1_logreg": "selection", "RF_importance": "selection", "LGBM_gain": "selection",
          "RFE_logreg": "selection",
          "PCA": "transformation", "Whiten": "transformation",
          "Center_L2": "transformation", "PLSDA": "transformation"}

In [ ]:

def r_full(Xtr, Xte, ytr):
    return Xtr, Xte


def r_anova(Xtr, Xte, ytr, k):
    s = SelectKBest(f_classif, k=min(k, Xtr.shape[1])).fit(Xtr, ytr)
    return s.transform(Xtr), s.transform(Xte)


def _mi_score(X, y):
    return mutual_info_classif(X, y, random_state=SEED)

def r_mi(Xtr, Xte, ytr, k):
    s = SelectKBest(_mi_score, k=min(k, Xtr.shape[1])).fit(Xtr, ytr)
    return s.transform(Xtr), s.transform(Xte)


def r_corr(Xtr, Xte, ytr, k):

    Y  = np.eye(3)[ytr]
    Xc = Xtr - Xtr.mean(0); Yc = Y - Y.mean(0)
    num = Xc.T @ Yc
    den = np.outer(np.sqrt((Xc**2).sum(0)), np.sqrt((Yc**2).sum(0))) + 1e-12
    corr = np.abs(num / den).max(1)
    idx = np.argsort(corr)[::-1][:min(k, Xtr.shape[1])]
    return Xtr[:, idx], Xte[:, idx]


def r_l1(Xtr, Xte, ytr, k):
    est = LogisticRegression(penalty="l1", solver="saga", C=0.05, max_iter=1500,
                             random_state=SEED, n_jobs=-1)
    s = SelectFromModel(est, max_features=min(k, Xtr.shape[1]),
                        threshold=-np.inf).fit(Xtr, ytr)
    return s.transform(Xtr), s.transform(Xte)


def r_rf(Xtr, Xte, ytr, k):
    est = RandomForestClassifier(n_estimators=200, max_depth=12, max_samples=0.7,
                                 min_samples_leaf=5, class_weight="balanced",
                                 random_state=SEED, n_jobs=-1)
    s = SelectFromModel(est, max_features=min(k, Xtr.shape[1]),
                        threshold=-np.inf).fit(Xtr, ytr)
    return s.transform(Xtr), s.transform(Xte)


def r_lgbm(Xtr, Xte, ytr, k):
    """LightGBM split-gain importance."""
    est = LGBMClassifier(n_estimators=300, max_depth=6, importance_type="gain",
                         class_weight="balanced", verbosity=-1,
                         random_state=SEED, n_jobs=-1)
    s = SelectFromModel(est, max_features=min(k, Xtr.shape[1]),
                        threshold=-np.inf).fit(Xtr, ytr)
    return s.transform(Xtr), s.transform(Xte)


def r_rfe(Xtr, Xte, ytr, k):
    k = min(k, Xtr.shape[1])
    if k >= Xtr.shape[1]:
        return Xtr, Xte
    est = LogisticRegression(C=0.1, max_iter=1000, solver="lbfgs", n_jobs=-1)
    r = RFE(est, n_features_to_select=k, step=0.15).fit(Xtr, ytr)
    return Xtr[:, r.support_], Xte[:, r.support_]


# --------------------------------------------------------------------------
# TRANSFORMATION : change the geometry of the same vectors
# --------------------------------------------------------------------------
def r_pca(Xtr, Xte, ytr, k=WHITEN_K):
    p = PCA(n_components=min(k, Xtr.shape[1], Xtr.shape[0]-1),
            whiten=False, random_state=SEED).fit(Xtr)
    return p.transform(Xtr), p.transform(Xte)


def r_whiten(Xtr, Xte, ytr, k=WHITEN_K):
    p = PCA(n_components=min(k, Xtr.shape[1], Xtr.shape[0]-1),
            whiten=True, random_state=SEED).fit(Xtr)
    return p.transform(Xtr), p.transform(Xte)


def r_center_l2(Xtr, Xte, ytr):
    """Centre on the training mean, then unit-norm -> dot product becomes cosine.
    Mean-pooled ProtT5 is anisotropic (large shared component); this is the
    transform that changes what SVM-RBF, kNN and the MLP actually see."""
    mu = Xtr.mean(0, keepdims=True)
    return normalize(Xtr - mu), normalize(Xte - mu)


def r_plsda(Xtr, Xte, ytr, k=PLS_K):
    Y = np.eye(3)[ytr]
    m = PLSRegression(n_components=min(k, Xtr.shape[1]), scale=False).fit(Xtr, Y)
    return m.transform(Xtr), m.transform(Xte)


REPRS = {
    "Full_1024": r_full,
    "ANOVA_f": r_anova, "MutualInfo": r_mi, "Corr_target": r_corr,
    "L1_logreg": r_l1, "RF_importance": r_rf, "LGBM_gain": r_lgbm,
    "RFE_logreg": r_rfe,
    "PCA": r_pca, "Whiten": r_whiten, "Center_L2": r_center_l2, "PLSDA": r_plsda,
}
# every method except the baseline is fitted with labels -> must be refit in-fold
SUPERVISED = {m for m in REPRS if m not in ("Full_1024", "PCA", "Whiten", "Center_L2")}

# methods whose subset/component size k is actually swept for evidence below,
# rather than fixed by convention. Center_L2 has no k (it keeps all 1024 dims
# and only changes geometry); Full_1024 has no k by definition.
K_SWEEP_METHODS = ["ANOVA_f", "MutualInfo", "Corr_target", "L1_logreg",
                   "RF_importance", "LGBM_gain", "RFE_logreg", "PCA", "Whiten", "PLSDA"]

In [ ]:

# ==============================================================================
PLS_K_GRID = [10, 20, 40, 60, 80, 120, 160]   
                                              

def probe_model():
    return LGBMClassifier(max_depth=6, num_leaves=31, n_estimators=200,
                          class_weight="balanced", verbosity=-1,
                          random_state=SEED, n_jobs=-1)


def sweep_k(method_name, grid, X_RAW, y, y_strat, cv):
    rfn = REPRS[method_name]
    rows = []
    for k in grid:
        accs, baccs = [], []
        for tr_idx, va_idx in cv.split(X_RAW, y_strat):
            Ztr, Zva = rfn(X_RAW[tr_idx], X_RAW[va_idx], y[tr_idx], k)
            sc = StandardScaler().fit(Ztr)
            m = probe_model().fit(sc.transform(Ztr), y[tr_idx])
            p = m.predict(sc.transform(Zva))
            accs.append(accuracy_score(y[va_idx], p))
            baccs.append(balanced_accuracy_score(y[va_idx], p))
        rows.append({"Method": method_name, "k": k,
                     "CV_ACC": np.mean(accs) * 100, "CV_BACC": np.mean(baccs) * 100})
    return pd.DataFrame(rows)


def run_k_selection(X_RAW, y, y_strat):
    cv_k = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    curves, best_k = [], {}
    print("=" * 78); print("  K-SELECTION SWEEP  (LightGBM probe, 5-fold CV)"); print("=" * 78)
    for m in K_SWEEP_METHODS:
        if m not in RUN_METHODS:
            continue
        grid = PLS_K_GRID if m == "PLSDA" else K_GRID
        t0 = time.time()
        curve = sweep_k(m, grid, X_RAW, y, y_strat, cv_k)
        curves.append(curve)
        k_star = int(curve.loc[curve["CV_ACC"].idxmax(), "k"])
        best_k[m] = k_star
        print(f"  {m:<14} " + "  ".join(f"k={r.k}:{r.CV_ACC:.1f}" for r in curve.itertuples())
              + f"   -> chosen k={k_star}   [{time.time()-t0:.0f}s]")
    curves_df = pd.concat(curves, ignore_index=True) if curves else pd.DataFrame()
    return best_k, curves_df

In [ ]:
# ==============================================================================
# MODEL FACTORY  -- 14 base models + 3 additions (KNN_cosine, SVM_Cosine,
# MLP_Deep)
# ==============================================================================
NEW_MODELS = {"KNN_cosine", "SVM_Cosine", "MLP_Deep"}

def get_ml_models():
    return {
        "Logistic Regression": LogisticRegression(C=1.0, max_iter=1000, solver="lbfgs",
            class_weight="balanced", random_state=SEED, n_jobs=-1),
        "Decision Tree": DecisionTreeClassifier(max_depth=5, min_samples_leaf=10,
            class_weight="balanced", random_state=SEED),
        "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=8,
            min_samples_leaf=5, class_weight="balanced", random_state=SEED, n_jobs=-1),
        "Extra Trees": ExtraTreesClassifier(n_estimators=200, max_depth=8,
            min_samples_leaf=5, class_weight="balanced", random_state=SEED, n_jobs=-1),
        "Gradient Boosting": HistGradientBoostingClassifier(max_iter=200, max_depth=3,
            learning_rate=0.05, class_weight="balanced", l2_regularization=0.1,
            random_state=SEED),
        "AdaBoost": AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1),
            n_estimators=100, random_state=SEED),
        "XGBoost": XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
            subsample=0.8, colsample_bytree=0.8, eval_metric="mlogloss",
            tree_method="hist", random_state=SEED, n_jobs=-1),
        "LightGBM": LGBMClassifier(max_depth=6, num_leaves=31, min_child_samples=20,
            n_estimators=200, reg_lambda=0.1, class_weight="balanced",
            verbosity=-1, random_state=SEED, n_jobs=-1),
        "CatBoost": CatBoostClassifier(iterations=200, depth=6, learning_rate=0.1,
            l2_leaf_reg=3, auto_class_weights="Balanced", verbose=0, random_seed=SEED),
        "SVM_RBF": SVC(C=1.0, kernel="rbf", probability=True,
            class_weight="balanced", random_state=SEED),
        "SVM_Linear": SVC(C=1.0, kernel="linear", probability=True,
            class_weight="balanced", random_state=SEED),
        "MLP": MLPClassifier(hidden_layer_sizes=(512, 256, 128), alpha=0.001,
            max_iter=500, early_stopping=True, validation_fraction=INNER_VAL_FRAC,
            random_state=SEED),
        # ------------------------------------------------------------- NEW
        "KNN_cosine": KNeighborsClassifier(n_neighbors=25, weights="distance",
            metric="cosine", n_jobs=-1),
        "SVM_Cosine": CalibratedClassifierCV(
            LinearSVC(C=0.5, class_weight="balanced", max_iter=5000, random_state=SEED),
            method="sigmoid", cv=3),
        "MLP_Deep": MLPClassifier(hidden_layer_sizes=(1024, 512, 256), alpha=3e-4,
            batch_size=256, learning_rate="adaptive", learning_rate_init=1e-3,
            max_iter=800, early_stopping=True, n_iter_no_change=25,
            validation_fraction=INNER_VAL_FRAC, random_state=SEED),
    }

DL_MODEL_NAMES = ["CNN", "DNN"]

In [ ]:
# ==============================================================================
# KERAS BASELINES
# ==============================================================================
import tensorflow as tf
from tensorflow.keras.models    import Sequential
from tensorflow.keras.layers    import (Dense, Dropout, Reshape,
                                        Conv1D, GlobalMaxPooling1D)
from tensorflow.keras.callbacks import EarlyStopping
tf.get_logger().setLevel("ERROR")

def build_dl(name, input_dim, n_classes):
    tf.keras.utils.set_random_seed(SEED)
    if name == "CNN":
        m = Sequential([Reshape((input_dim, 1), input_shape=(input_dim,)),
            Conv1D(64, 5, activation="relu"), Conv1D(32, 3, activation="relu"),
            GlobalMaxPooling1D(), Dense(64, activation="relu"), Dropout(0.3),
            Dense(n_classes, activation="softmax")])
    else:
        m = Sequential([Dense(512, activation="relu", input_shape=(input_dim,)),
            Dropout(0.3), Dense(256, activation="relu"), Dropout(0.3),
            Dense(128, activation="relu"), Dropout(0.2),
            Dense(n_classes, activation="softmax")])
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
    return m

In [ ]:
# ==============================================================================
# LOAD  (training split only)
# ==============================================================================
df = pd.read_csv(FEATURES_PATH)
missing = [c for c in META_COLS if c not in df.columns]
if missing:
    raise ValueError(f"Missing metadata column(s): {missing}")

dim_cols  = [c for c in df.columns if c.startswith("dim_")]
X_RAW     = df[dim_cols].to_numpy(np.float32)
y         = df[TARGET_COL].to_numpy()
y_strat   = df[STRATIFY_COL].to_numpy()
N_CLASSES = int(len(np.unique(y)))

print("=" * 78)
print("  FEATURE OPTIMIZATION : selection vs transformation, 17 models, 5-fold CV")
print("=" * 78)
print(f"  X {X_RAW.shape}   class counts {np.bincount(y)}")
print(f"  Methods : {RUN_METHODS}")
print("  Subset/component size k : swept per method below, not fixed by convention")
print("  Locked external split (features_prott5_2660.csv) : NOT read here")

if X_RAW.shape[1] < 500:
    raise ValueError("This file is already reduced. Point FEATURES_PATH at the "
                     "FULL 1024-dim ProtT5 file -- every method is applied in-fold.")
unknown = [t for t in RUN_METHODS if t not in REPRS]
if unknown:
    raise ValueError(f"Unknown method(s) {unknown}. Available: {list(REPRS)}")

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
all_results = []

In [ ]:
# ==============================================================================
# METRICS
# ==============================================================================

def get_metrics(y_true, y_pred, y_score=None, n_classes=3):
    m = {
        "acc" : accuracy_score(y_true, y_pred),
        "bacc": balanced_accuracy_score(y_true, y_pred),
        "f1"  : f1_score(y_true, y_pred, average="macro"),
        "pre" : precision_score(y_true, y_pred, average="macro", zero_division=0),
        "sens": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "mcc" : matthews_corrcoef(y_true, y_pred),
    }
    cm = confusion_matrix(y_true, y_pred, labels=list(range(n_classes)))
    specs, npvs = [], []
    for i in range(n_classes):
        tp = cm[i, i]; fp = cm[:, i].sum() - tp; fn = cm[i, :].sum() - tp
        tn = cm.sum() - (tp + fp + fn)
        specs.append(tn / (tn + fp) if (tn + fp) else 0.0)
        npvs.append(tn / (tn + fn) if (tn + fn) else 0.0)
        m[f"sens_c{i}"] = tp / (tp + fn) if (tp + fn) else 0.0
        m[f"spec_c{i}"] = specs[-1]
        m[f"pre_c{i}"]  = tp / (tp + fp) if (tp + fp) else 0.0
        m[f"npv_c{i}"]  = npvs[-1]
    m["spec"] = float(np.mean(specs))
    m["npv"]  = float(np.mean(npvs))
    if y_score is not None:
        yb = label_binarize(y_true, classes=list(range(n_classes)))
        try:
            m["auc"] = roc_auc_score(y_true, y_score, multi_class="ovr",
                                     average="macro", labels=list(range(n_classes)))
        except Exception:
            m["auc"] = np.nan
        try:
            m["ap"] = average_precision_score(yb, y_score, average="macro")
        except Exception:
            m["ap"] = np.nan
    else:
        m["auc"] = m["ap"] = np.nan
    return m


def fmt(a, raw=False):
    a = np.asarray(a, float); a = a[~np.isnan(a)]
    if a.size == 0:
        return "n/a"
    s = 1 if raw else 100
    return f"{a.mean()*s:.2f} \u00b1 {a.std()*s:.2f}"


def fmt_ci(a, raw=False):
    """Mean with a 95% CI across folds (normal approximation on the fold SE)."""
    a = np.asarray(a, float); a = a[~np.isnan(a)]
    if a.size == 0:
        return "n/a"
    s  = 1 if raw else 100
    mu = a.mean() * s
    se = a.std(ddof=1) / np.sqrt(a.size) * s if a.size > 1 else 0.0
    return f"{mu:.2f} [{mu-1.96*se:.2f}, {mu+1.96*se:.2f}]"


METRIC_KEYS   = ["acc","bacc","f1","sens","spec","pre","npv","mcc","auc","ap"]
PERCLASS_KEYS = [f"{s}_c{i}" for i in range(3) for s in ("sens","spec","pre","npv")]


def new_store():
    d = {f"tr_{k}": [] for k in METRIC_KEYS}
    d.update({f"cv_{k}": [] for k in METRIC_KEYS + PERCLASS_KEYS})
    return d


def record(store, y_tr, y_ts, p_tr, p_ts, s_ts=None):
    a = get_metrics(y_tr, p_tr)
    b = get_metrics(y_ts, p_ts, s_ts)
    for k in METRIC_KEYS:
        store[f"tr_{k}"].append(a[k])
    for k in METRIC_KEYS + PERCLASS_KEYS:
        store[f"cv_{k}"].append(b[k])


def build_row(name, stage, store, best_solo=None, extra=None):
    tr  = np.mean(store["tr_acc"]) * 100
    cvv = np.mean(store["cv_acc"]) * 100
    row = {"Config": name, "Stage": stage}
    for k, lbl in [("acc","ACC"),("bacc","BACC"),("sens","SENS"),("spec","SPEC"),
                   ("pre","PRE"),("npv","NPV"),("f1","F1"),("mcc","MCC"),
                   ("auc","AUC"),("ap","AP")]:
        raw = k in ("mcc","auc","ap")
        row[f"Train_{lbl}"]   = fmt(store[f"tr_{k}"], raw)
        row[f"CV_{lbl}"]      = fmt(store[f"cv_{k}"], raw)
        row[f"CV_{lbl}_95CI"] = fmt_ci(store[f"cv_{k}"], raw)
    for i in range(3):
        cn = CLASS_NAMES[i]
        for s, lbl in [("sens","SENS"),("spec","SPEC"),("pre","PRE"),("npv","NPV")]:
            row[f"CV_{lbl}_{cn}"] = fmt(store[f"cv_{s}_c{i}"])
    row["CV_ACC_raw"]  = round(cvv, 4)
    row["CV_BACC_raw"] = round(np.mean(store["cv_bacc"]) * 100, 4)
    row["CV_F1_raw"]   = round(np.mean(store["cv_f1"]) * 100, 4)
    row["Overfit_Gap"] = round(tr - cvv, 2)
    if best_solo is not None:
        row["Best_Solo_ACC"] = round(best_solo, 2)
        row["Gain_vs_Best"]  = round(cvv - best_solo, 2)
    if extra:
        row.update(extra)
    return row

In [ ]:
# ==============================================================================
# RUN THE K-SELECTION SWEEP NOW, ONCE, BEFORE THE MAIN 17-MODEL COMPARISON
# ==============================================================================
BEST_K, K_CURVES = run_k_selection(X_RAW, y, y_strat)
if not K_CURVES.empty:
    K_CURVES.to_csv(os.path.join(OUTPUT_DIR, "k_selection_sweep.csv"), index=False)

print(f"\n{'='*78}\n  K CHOSEN PER METHOD (evidence-based, not a shared convention)\n{'='*78}")
for m in RUN_METHODS:
    if m in BEST_K:
        print(f"  {m:<14} k = {BEST_K[m]}")
    elif m not in ("Full_1024", "Center_L2"):
        print(f"  {m:<14} k = (uses its own default -- not in this sweep)")
print("  Center_L2 keeps all 1024 dims (geometry-only transform, no size choice).")
print("  Full_1024 is the untouched baseline, k = 1024 by definition.")

In [ ]:
# ==============================================================================
# MAIN LOOP -- representation fitted once per (method, fold) at ITS chosen k,
# shared by all 17 models
# ==============================================================================
SAMPLE_WEIGHT_MODELS = {"XGBoost", "AdaBoost"}

for mname in RUN_METHODS:
    fam = FAMILY[mname]
    tag = "[refit in-fold]" if mname in SUPERVISED else "[unsupervised]"
    k_used = BEST_K.get(mname, None)
    k_label = f", k={k_used}" if k_used is not None else ""
    print(f"\n{'-'*78}\n  {mname}   ({fam}{k_label})   {tag}\n{'-'*78}")
    rfn = REPRS[mname]

    t0, folds = time.time(), []
    for tr_idx, va_idx in cv.split(X_RAW, y_strat):
        if k_used is not None:
            Ztr, Zva = rfn(X_RAW[tr_idx], X_RAW[va_idx], y[tr_idx], k_used)
        else:
            Ztr, Zva = rfn(X_RAW[tr_idx], X_RAW[va_idx], y[tr_idx])
        sc = StandardScaler().fit(Ztr)
        folds.append((sc.transform(Ztr), sc.transform(Zva), y[tr_idx], y[va_idx]))
    input_dim = folds[0][0].shape[1]
    print(f"  {input_dim} features   (representation fitted {N_SPLITS}x, {time.time()-t0:.0f}s)")

    all_models = dict(get_ml_models())
    all_models.update({n: "DL" for n in DL_MODEL_NAMES})

    for model_name, model in all_models.items():
        print(f"  -> {model_name:<22}", end=" ", flush=True)
        store = new_store()
        for fold, (X_tr, X_va, y_tr, y_va) in enumerate(folds):
            sample_w = compute_sample_weight(class_weight="balanced", y=y_tr)
            if model == "DL":
                rs = np.random.RandomState(SEED + fold); perm = rs.permutation(len(y_tr))
                cut = int(len(perm) * (1 - INNER_VAL_FRAC))
                i_fit, i_val = perm[:cut], perm[cut:]
                net = build_dl(model_name, X_tr.shape[1], N_CLASSES)
                cw = compute_class_weight("balanced", classes=np.unique(y_tr), y=y_tr)
                net.fit(X_tr[i_fit], y_tr[i_fit],
                        validation_data=(X_tr[i_val], y_tr[i_val]),
                        epochs=100, batch_size=64, verbose=0,
                        class_weight={int(c): w for c, w in zip(np.unique(y_tr), cw)},
                        callbacks=[EarlyStopping(patience=10, restore_best_weights=True)])
                s_tr = net.predict(X_tr, verbose=0); s_va = net.predict(X_va, verbose=0)
                tf.keras.backend.clear_session()
            else:
                from sklearn.base import clone
                est = clone(model)
                if model_name in SAMPLE_WEIGHT_MODELS:
                    est.fit(X_tr, y_tr, sample_weight=sample_w)
                else:
                    est.fit(X_tr, y_tr)
                s_tr = est.predict_proba(X_tr); s_va = est.predict_proba(X_va)
            record(store, y_tr, y_va, np.argmax(s_tr, 1), np.argmax(s_va, 1), s_va)

        row = build_row(model_name, mname, store,
                        extra={"Method": mname, "Family": fam, "N_Features": input_dim,
                               "Model": model_name, "New_Model": model_name in NEW_MODELS})
        all_results.append(row)
        print(f"ACC {row['CV_ACC_raw']:.2f}  BACC {row['CV_BACC_raw']:.2f}  "
              f"F1 {row['CV_F1_raw']:.2f}  gap {row['Overfit_Gap']:.1f}")
        gc.collect()

In [ ]:
# ==============================================================================
# RESULTS
# ==============================================================================
res = (pd.DataFrame(all_results)
         .sort_values(["Method", "CV_ACC_raw"], ascending=[True, False])
         .reset_index(drop=True))
res.to_csv(os.path.join(OUTPUT_DIR, "FeatureOpt_selection_vs_transform_17ML.csv"), index=False)

print(f"\n{'='*78}\n  FULL TABLE\n{'='*78}")
print(res[["Method","Family","N_Features","Model","New_Model","CV_ACC","CV_BACC",
           "CV_F1","CV_MCC","Overfit_Gap"]].to_string(index=False))

print(f"\n{'='*78}\n  METHOD RANKING  (mean over all 17 models)\n{'='*78}")
rank = (res.groupby("Method")
          .agg(family=("Family","first"), n_features=("N_Features","first"),
               mean_ACC=("CV_ACC_raw","mean"), best_ACC=("CV_ACC_raw","max"),
               best_model=("CV_ACC_raw", lambda s: res.loc[s.idxmax(),"Model"]),
               mean_BACC=("CV_BACC_raw","mean"), mean_F1=("CV_F1_raw","mean"))
          .sort_values("mean_ACC", ascending=False).round(3))
print(rank.to_string())
rank.to_csv(os.path.join(OUTPUT_DIR, "FeatureOpt_method_ranking.csv"))

if "Full_1024" in rank.index:
    base = rank.loc["Full_1024", "mean_ACC"]
    print(f"\n  Delta vs Full_1024 baseline (mean CV accuracy = {base:.3f}):")
    for t, r in rank.iterrows():
        if t != "Full_1024":
            print(f"    {t:<16} {r['family']:<14} {int(r['n_features']):>4}f  "
                  f"{r['mean_ACC']:.3f}  ({r['mean_ACC']-base:+.3f})")

print(f"\n{'='*78}\n  FAMILY SUMMARY  -- the sentence that goes in the manuscript\n{'='*78}")
fam = res.groupby("Family").agg(mean_ACC=("CV_ACC_raw","mean"),
                                best_ACC=("CV_ACC_raw","max")).round(3)
print(fam.to_string())
print("""
  Expected pattern on a dense PLM embedding, and how to report it:
    * every SELECTION method lands within ~1 point of Full_1024 and of each
      other -> report as one finding: "standard univariate, embedded and wrapper
      selection did not improve on the full embedding, consistent with the
      distributed nature of PLM representations."  Do NOT report seven near-equal
      rows as seven results.
    * TRANSFORMATION methods differ from each other more than the selectors do,
      because they change the metric rather than the coordinate set. The best of
      them is the honest headline of this stage.""")

print(f"\n{'='*78}\n  TOP 3 (model, method) PAIRS PER METHOD  -> candidate stacking bases\n{'='*78}")
for m in res["Method"].unique():
    sub = res[res.Method == m].nlargest(3, "CV_ACC_raw")
    print(f"  {m:<16} " + " | ".join(f"{r.Model} {r.CV_ACC_raw:.2f}" for r in sub.itertuples()))
print("""
  Carry into notebook 2 the highest-accuracy (model, representation) pairs that
  span DIFFERENT representations -- diversity of geometry helps a stack more than
  the last fraction of a point from any single member. On this ProtT5 file expect
  those pairs to be SVM/MLP on Whiten, the trees on an importance/RFE subset, and
  KNN/LR on Center_L2.""")

print(f"\n{'='*78}\n  SUBSET/COMPONENT SIZE k  -- chosen by evidence, per method\n{'='*78}")
for m, k in BEST_K.items():
    print(f"  {m:<14} k = {k}   (see k_selection_sweep.csv for the full curve)")
print("""
  Report k this way in Methods: "the subset/component size for each selection
  or transformation method was chosen by a 5-fold CV sweep over
  {32,64,128,256,384,512,768} using a LightGBM probe, rather than fixed by
  convention" -- with the sweep CSV as supplementary evidence. That is a
  materially stronger answer to "why k=256?" than citing the prior notebook's
  default, and it is what this notebook now actually does.""")